In [8]:
import pandas as pd
from edgar.shared import AppLogger,read_csv,read_csv_incrementally
from edgar.config import Config
config = Config()
logger = AppLogger()


In [ ]:
# raw_companies inspection
df_raw_companies = read_csv(logger,config.raw_dir / "raw_companies.csv")
df_raw_companies.info()

2026-05-29 09:44:58,251 | INFO | read_csv: 987 rows × 14 cols ← raw_companies.csv


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 987 entries, 0 to 986
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   cik                              987 non-null    int64  
 1   name                             987 non-null    object 
 2   entityType                       987 non-null    object 
 3   sic                              986 non-null    float64
 4   sicDescription                   986 non-null    object 
 5   category                         986 non-null    object 
 6   fiscalYearEnd                    986 non-null    float64
 7   stateOfIncorporation             904 non-null    object 
 8   stateOfIncorporationDescription  904 non-null    object 
 9   ein                              987 non-null    int64  
 10  lei                              3 non-null      object 
 11  tickers                          987 non-null    object 
 12  exchanges             

In [ ]:
# raw_filings inspection
df_raw_filings = read_csv(logger,config.raw_dir / "raw_filings.csv")
df_raw_filings.info()

C:\SWE\GithubRepos\Edgar\src\edgar\shared\csv.py:8: DtypeWarning: Columns (6,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **kwargs)
2026-05-29 09:46:02,297 | INFO | read_csv: 970,892 rows × 17 cols ← raw_filings.csv


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 970892 entries, 0 to 970891
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   cik                    970892 non-null  int64  
 1   accessionNumber        970892 non-null  object 
 2   filingDate             970892 non-null  object 
 3   reportDate             722539 non-null  object 
 4   acceptanceDateTime     970892 non-null  object 
 5   form                   970892 non-null  object 
 6   act                    402375 non-null  object 
 7   core_type              970891 non-null  object 
 8   fileNumber             404666 non-null  object 
 9   filmNumber             401806 non-null  object 
 10  items                  115879 non-null  object 
 11  primaryDocDescription  808924 non-null  object 
 12  primaryDocument        970411 non-null  object 
 13  size                   970892 non-null  int64  
 14  isXBRL                 970892 non-nu

In [19]:
# raw_company_facts inspection — stream ALL chunks, accumulate stats (never full load)
path = config.raw_dir / "raw_company_facts.csv"

total_rows = 0
non_null = None          # Series: non-null count per column
dtypes = None            # captured from first chunk
num = {}                 # col -> [n, sum, sumsq, min, max] for numeric cols

for chunk in read_csv_incrementally(logger, path):
    total_rows += len(chunk)
    nn = chunk.notnull().sum()
    non_null = nn if non_null is None else non_null + nn
    if dtypes is None:
        dtypes = chunk.dtypes
    for col in chunk.select_dtypes("number"):
        s = chunk[col].dropna()
        if s.empty:
            continue
        if col not in num:
            num[col] = [s.size, s.sum(), (s ** 2).sum(), s.min(), s.max()]
        else:
            st = num[col]
            st[0] += s.size; st[1] += s.sum(); st[2] += (s ** 2).sum()
            st[3] = min(st[3], s.min()); st[4] = max(st[4], s.max())

print(f"total rows: {total_rows:,}")

# info()-equivalent
info = pd.DataFrame({"dtype": dtypes, "non_null": non_null,
                     "null": total_rows - non_null})
print(info)

# describe()-equivalent for numeric cols (quantiles omitted — not cheap to stream)
desc = pd.DataFrame(
    [[c, n, s / n, (sq / n - (s / n) ** 2) ** 0.5, mn, mx]
     for c, (n, s, sq, mn, mx) in num.items()],
    columns=["col", "count", "mean", "std", "min", "max"],
).set_index("col")
print(desc)

2026-05-29 09:55:31,690 | INFO | read_csv_incrementally: ← raw_company_facts.csv (chunk_size=1,000,000)
C:\Users\samet\AppData\Local\Temp\ipykernel_31616\1475752337.py:23: RuntimeWarning: overflow encountered in scalar add
  st[0] += s.size; st[1] += s.sum(); st[2] += (s ** 2).sum()


total rows: 21,782,145
            dtype  non_null      null
cik         int64  21782145         0
taxonomy   object  21782145         0
concept    object  21782145         0
unit       object  21782145         0
val       float64  21782145         0
start      object  13501438   8280707
end        object  21782145         0
fy        float64  21644399    137746
fp         object  21628359    153786
form       object  21782145         0
filed      object  21782145         0
accn       object  21782145         0
frame      object   8943963  12838182
        count          mean           std           min           max
col                                                                  
cik  21782145  8.500417e+05           NaN  1.800000e+03  2.082866e+06
val  21782145  1.820840e+12  6.743524e+15 -9.100000e+12  3.135437e+19
fy   21644399  2.017170e+03  4.901266e+01  0.000000e+00  2.026000e+03


C:\Users\samet\AppData\Local\Temp\ipykernel_31616\1475752337.py:35: RuntimeWarning: invalid value encountered in scalar power
  [[c, n, s / n, (sq / n - (s / n) ** 2) ** 0.5, mn, mx]


In [20]:
chunks = read_csv_incrementally(logger, config.raw_dir / "raw_company_facts.csv")
first = next(chunks)
first.info()
first.describe()
first.head()

2026-05-29 10:27:20,421 | INFO | read_csv_incrementally: ← raw_company_facts.csv (chunk_size=1,000,000)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 13 columns):
 #   Column    Non-Null Count    Dtype  
---  ------    --------------    -----  
 0   cik       1000000 non-null  int64  
 1   taxonomy  1000000 non-null  object 
 2   concept   1000000 non-null  object 
 3   unit      1000000 non-null  object 
 4   val       1000000 non-null  float64
 5   start     618919 non-null   object 
 6   end       1000000 non-null  object 
 7   fy        997673 non-null   float64
 8   fp        994166 non-null   object 
 9   form      1000000 non-null  object 
 10  filed     1000000 non-null  object 
 11  accn      1000000 non-null  object 
 12  frame     410799 non-null   object 
dtypes: float64(2), int64(1), object(10)
memory usage: 99.2+ MB


,cik,taxonomy,concept,unit,val,start,end,fy,fp,form,filed,accn,frame
0,1800,dei,EntityCommonStockSharesOutstanding,shares,1.545912e+09,NaN,2009-06-30,2009.0,Q2,10-Q,2009-08-07,0001104659-09-048013,CY2009Q2I
1,1800,dei,EntityCommonStockSharesOutstanding,shares,1.546738e+09,NaN,2009-09-30,2009.0,Q3,10-Q,2009-11-06,0001104659-09-063192,CY2009Q3I
2,1800,dei,EntityCommonStockSharesOutstanding,shares,1.552643e+09,NaN,2010-01-31,2009.0,FY,10-K,2010-02-19,0001047469-10-001018,CY2009Q4I
3,1800,dei,EntityCommonStockSharesOutstanding,shares,1.543565e+09,NaN,2010-03-31,2010.0,Q1,10-Q,2010-05-04,0001104659-10-033097,CY2010Q1I
4,1800,dei,EntityCommonStockSharesOutstanding,shares,1.544029e+09,NaN,2010-06-30,2010.0,Q2,10-Q,2010-08-05,0001104659-10-042282,CY2010Q2I
